# Build all-case CNV matrices from annotated CNR files with TFx metadata

This notebook builds all-case longitudinal CNV input files from per-sample `*.annotated.cnr` files and merges TFx / histology / receptor metadata from:

`Metasheet_Analysis.with_Histology_Receptor_status.xlsx`

Outputs are saved under:

`/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/`

Main outputs:
- `All_representative_patients_longitudinal_gene_CNV_state_matrix.tsv`
- `All_representative_patients_longitudinal_CNV_sample_annotation.tsv`
- `All_representative_patients_longitudinal_CNR_manifest.tsv`

Important: `Tumor_Fraction` is kept as the original fraction, and `Tumor_Fraction_percent` is also created.


============================================================
Build all-case longitudinal CNV gene-state matrix from
per-sample *.annotated.cnr files and add TFx / histology /
receptor metadata from Metasheet_Analysis.with_Histology_Receptor_status.xlsx


In [1]:
# ============================================================
#
# Input CNV files:
#   /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/Annotated_cnr_file/*.annotated.cnr
#
# Input metadata file:
#   /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.with_Histology_Receptor_status.xlsx
#
# Outputs:
#   /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/
#
#   All_representative_patients_longitudinal_gene_CNV_state_matrix.tsv
#   All_representative_patients_longitudinal_CNV_sample_annotation.tsv
#   All_representative_patients_longitudinal_CNR_manifest.tsv
#
# Important:
#   Tumor_Fraction in the metasheet appears as a fraction, e.g. 0.09397.
#   This script saves:
#       Tumor_Fraction          = original fraction value
#       Tumor_Fraction_percent  = Tumor_Fraction * 100 when value <= 1
#


In [15]:
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import sys

!{sys.executable} -m pip install --user --upgrade "openpyxl>=3.1.5"



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [16]:
# Path
# ============================================================

BASE_DIR = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA")

cnr_dir = BASE_DIR / "CNVKit/Annotated_cnr_file"

metasheet_file = BASE_DIR / "Clinical_data/Metasheet_Analysis.with_Histology_Receptor_status.xlsx"

out_dir = BASE_DIR / "CNVKit/All_case_CNV_extraction"
out_dir.mkdir(parents=True, exist_ok=True)

out_matrix = out_dir / "All_representative_patients_longitudinal_gene_CNV_state_matrix.tsv"
out_annotation = out_dir / "All_representative_patients_longitudinal_CNV_sample_annotation.tsv"
out_manifest = out_dir / "All_representative_patients_longitudinal_CNR_manifest.tsv"


In [18]:
# 2. Helper functions
# ============================================================

def clean_blank(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() in {"na", "nan", "none", "null"}:
        return pd.NA

    return s


def extract_sample_id_from_cnr_path(path):
    """
    ARWN-NB4.annotated.cnr -> ARWN-NB4
    """
    name = Path(path).name
    name = re.sub(r"\.annotated\.cnr$", "", name)
    return name


def normalize_collaborator_sample_id(x):
    """
    AUP-AL1Y-NB1 -> AL1Y-NB1
    AL1Y-NB1     -> AL1Y-NB1
    """
    x = clean_blank(x)
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    m = re.search(r"\bAUP-([A-Z0-9]{4}-NB\d+)\b", s)
    if m:
        return m.group(1)

    m = re.search(r"\b([A-Z0-9]{4}-NB\d+)\b", s)
    if m:
        return m.group(1)

    return pd.NA


def extract_patient_identifier(x):
    """
    Extract 4-character AURORA patient identifier.

    Examples:
      AUP-AQ0Z-TTP2-B -> AQ0Z
      AQ0Z-NB1        -> AQ0Z
      ARWN-NB4        -> ARWN
    """
    x = clean_blank(x)
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    m = re.search(r"\bAUP-([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    m = re.search(r"\b([A-Z0-9]{4})-NB\d+\b", s)
    if m:
        return m.group(1)

    m = re.search(r"\b([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    return pd.NA


def extract_draw_num(x):
    x = clean_blank(x)
    if pd.isna(x):
        return np.nan

    m = re.search(r"NB(\d+)", str(x).upper())
    if not m:
        return np.nan

    return int(m.group(1))


def normalize_cnv_state(x):
    """
    Standardize CNV call/state values.
    """
    x = clean_blank(x)
    if pd.isna(x):
        return "NEUT"

    s = str(x).strip().upper()

    if s in {"NEUT", "NEUTRAL", "DIPLOID", "NORMAL", "2"}:
        return "NEUT"

    if s in {"GAIN", "GAINS"}:
        return "GAIN"

    if s in {"AMP", "AMPLIFICATION", "AMPLIFIED"}:
        return "AMP"

    if s in {"HLAMP", "HIGH_LEVEL_AMP", "HIGH-LEVEL-AMP", "HIGH_LEVEL_AMPLIFICATION"}:
        return "HLAMP"

    if s in {"HETD", "HETDEL", "HET_DEL", "HETEROZYGOUS_DELETION"}:
        return "HETD"

    if s in {"HOMD", "HOMDEL", "HOM_DEL", "HOMOZYGOUS_DELETION"}:
        return "HOMD"

    if s in {"LOSS", "DEL", "DELETION"}:
        return "HETD"

    return s


state_priority = {
    "NEUT": 0,
    "HETD": 1,
    "HOMD": 2,
    "GAIN": 3,
    "AMP": 4,
    "HLAMP": 5,
}


def get_state_priority(state):
    return state_priority.get(str(state).upper(), 1)


def detect_call_column(df, sample_id):
    """
    Prefer corrected CNV call, then event.
    Expected examples:
      ARWN-NB4.Corrected_Call
      ARWN-NB4.event
    """
    preferred = [
        f"{sample_id}.Corrected_Call",
        f"{sample_id}.event",
        "Corrected_Call",
        "event",
    ]

    for c in preferred:
        if c in df.columns:
            return c

    corrected_candidates = [c for c in df.columns if c.endswith(".Corrected_Call")]
    if corrected_candidates:
        return corrected_candidates[0]

    event_candidates = [c for c in df.columns if c.endswith(".event")]
    if event_candidates:
        return event_candidates[0]

    raise ValueError(
        f"Could not find CNV call column for sample {sample_id}. "
        f"Columns include: {list(df.columns)[:20]}"
    )


def parse_gene_list(x):
    x = clean_blank(x)
    if pd.isna(x):
        return []

    genes = []
    for g in str(x).split(","):
        g = g.strip()
        if g == "" or g.upper() in {"NA", "NAN", "NONE", "NULL", "."}:
            continue
        genes.append(g)

    return genes


def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def summarize_one_cnr(cnr_path):
    """
    Read one annotated CNR and return:
      gene_state_series: index gene, value collapsed CNV state
      manifest row
      annotation row
    """
    sample_id = extract_sample_id_from_cnr_path(cnr_path)
    patient_identifier = extract_patient_identifier(sample_id)
    draw_num = extract_draw_num(sample_id)

    df = pd.read_csv(
        cnr_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_values=[],
        encoding="latin1"
    )

    if "gene" not in df.columns:
        raise ValueError(f"Missing required 'gene' column in {cnr_path}")

    call_col = detect_call_column(df, sample_id)

    tmp = df[["gene", call_col]].copy()
    tmp = tmp.rename(columns={call_col: "cnv_state"})
    tmp["cnv_state"] = tmp["cnv_state"].apply(normalize_cnv_state)
    tmp["gene"] = tmp["gene"].apply(parse_gene_list)

    tmp = tmp.explode("gene")
    tmp = tmp[tmp["gene"].notna() & (tmp["gene"].astype(str).str.strip() != "")].copy()

    tmp["state_rank"] = tmp["cnv_state"].apply(get_state_priority)

    # For each gene in this sample, keep the strongest altered state.
    tmp = tmp.sort_values(["gene", "state_rank"], ascending=[True, False])

    gene_state = (
        tmp
        .drop_duplicates(subset=["gene"], keep="first")
        .set_index("gene")["cnv_state"]
        .sort_index()
    )

    n_unique_genes = int(gene_state.shape[0])
    n_altered_genes = int((gene_state != "NEUT").sum())
    n_amp_or_hlamp = int(gene_state.isin(["AMP", "HLAMP"]).sum())
    n_hlamp = int((gene_state == "HLAMP").sum())

    manifest_row = {
        "sample_id": sample_id,
        "Patient_Identifier": patient_identifier,
        "draw_num": draw_num,
        "cnr_file": str(cnr_path),
        "call_column_used": call_col,
        "n_cnr_rows": int(df.shape[0]),
        "n_gene_rows_after_explode": int(tmp.shape[0]),
        "n_unique_genes": n_unique_genes,
        "n_altered_genes_non_NEUT": n_altered_genes,
        "n_amp_or_hlamp_genes": n_amp_or_hlamp,
        "n_hlamp_genes": n_hlamp,
    }

    annotation_row = {
        "sample_id": sample_id,
        "Patient_Identifier": patient_identifier,
        "draw_num": draw_num,
        "CNR_file": str(cnr_path),
        "CNV_burden_non_NEUT_genes": n_altered_genes,
        "n_amp_or_hlamp_genes": n_amp_or_hlamp,
        "n_hlamp_genes": n_hlamp,
    }

    return gene_state, manifest_row, annotation_row


In [19]:
# 3. Read metasheet for TFx / histology / receptor annotation
# ============================================================

if not metasheet_file.exists():
    raise FileNotFoundError(f"Metasheet not found: {metasheet_file}")

metasheet = pd.read_excel(metasheet_file, dtype=str, engine="openpyxl")

# The file may have either exact or slightly changed column names.
collab_col = first_existing_column(
    metasheet,
    ["Collaborator_sample_id", "Collaborator Sample ID", "Barcode"]
)
broad_col = first_existing_column(
    metasheet,
    ["Broad_entity:sample_id", "Broad_entity_sample_id", "Broad_sample_id"]
)
patient_id_col = first_existing_column(
    metasheet,
    ["Patient_id", "patient_id", "Patient_ID"]
)
patient_identifier_col = first_existing_column(
    metasheet,
    ["Patient_Identifier", "Patient Identifier"]
)

if collab_col is None:
    raise ValueError(
        "Could not find Collaborator_sample_id column in metasheet. "
        f"Available columns: {list(metasheet.columns)}"
    )

metasheet["sample_id"] = metasheet[collab_col].apply(normalize_collaborator_sample_id)

if patient_identifier_col is not None:
    metasheet["Patient_Identifier_from_metasheet"] = metasheet[patient_identifier_col].apply(clean_blank)
else:
    metasheet["Patient_Identifier_from_metasheet"] = metasheet["sample_id"].apply(extract_patient_identifier)

# Tumor_Fraction is stored as a fraction in the metasheet.
if "Tumor_Fraction" in metasheet.columns:
    metasheet["Tumor_Fraction"] = pd.to_numeric(metasheet["Tumor_Fraction"], errors="coerce")
else:
    metasheet["Tumor_Fraction"] = np.nan

metasheet["Tumor_Fraction_percent"] = np.where(
    metasheet["Tumor_Fraction"].notna() & (metasheet["Tumor_Fraction"] <= 1),
    metasheet["Tumor_Fraction"] * 100,
    metasheet["Tumor_Fraction"]
)

# Numeric metadata
for c in [
    "Mean_coverage",
    "Ploidy",
    "Subclone Fraction",
    "Frac. Genome Subclonal",
    "Frac. CNA Subclonal",
    "Frac_Subnucleosomal_90_150",
    "Frac_Mononucleosomal_151_250",
    "Frac_Dinucleosomal_251_450",
    "Frac_Trinucleosomal_451_650",
]:
    if c in metasheet.columns:
        metasheet[c] = pd.to_numeric(metasheet[c], errors="coerce")

# Keep useful columns if present.
keep_cols = ["sample_id", "Patient_Identifier_from_metasheet", "Tumor_Fraction", "Tumor_Fraction_percent"]

if broad_col is not None:
    keep_cols.append(broad_col)

if patient_id_col is not None:
    keep_cols.append(patient_id_col)

for c in [
    "Histological_subtype",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Molecular_subtype",
    "Mean_coverage",
    "Ploidy",
    "Subclone Fraction",
    "Frac. Genome Subclonal",
    "Frac. CNA Subclonal",
    "Frac_Subnucleosomal_90_150",
    "Frac_Mononucleosomal_151_250",
    "Frac_Dinucleosomal_251_450",
    "Frac_Trinucleosomal_451_650",
]:
    if c in metasheet.columns:
        keep_cols.append(c)

metasheet_annot = metasheet[keep_cols].drop_duplicates(subset=["sample_id"], keep="first").copy()

print("Metasheet rows:", metasheet.shape[0])
print("Metasheet samples with usable sample_id:", metasheet_annot["sample_id"].notna().sum())
print("Metasheet samples with Tumor_Fraction:", metasheet_annot["Tumor_Fraction"].notna().sum())


Metasheet rows: 784
Metasheet samples with usable sample_id: 784
Metasheet samples with Tumor_Fraction: 784


In [20]:
# 4. Find input CNR files
# ============================================================

cnr_files = sorted(cnr_dir.glob("*.annotated.cnr"))

if len(cnr_files) == 0:
    raise FileNotFoundError(f"No *.annotated.cnr files found in: {cnr_dir}")

print("\nAnnotated CNR directory:", cnr_dir)
print("Annotated CNR files found:", len(cnr_files))
print("First 5 files:")
for p in cnr_files[:5]:
    print("  ", p.name)



Annotated CNR directory: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/Annotated_cnr_file
Annotated CNR files found: 784
First 5 files:
   AL1Y-NB1.annotated.cnr
   AL1Y-NB2.annotated.cnr
   AL1Y-NB3.annotated.cnr
   AL1Y-NB4.annotated.cnr
   AL1Y-NB5.annotated.cnr


In [21]:
# 5. Process all annotated CNR files
# ============================================================

gene_state_by_sample = {}
manifest_rows = []
annotation_rows = []

for i, cnr_path in enumerate(cnr_files, start=1):
    if i % 25 == 0 or i == 1 or i == len(cnr_files):
        print(f"[{i}/{len(cnr_files)}] {cnr_path.name}")

    gene_state, manifest_row, annotation_row = summarize_one_cnr(cnr_path)

    sample_id = manifest_row["sample_id"]
    gene_state_by_sample[sample_id] = gene_state
    manifest_rows.append(manifest_row)
    annotation_rows.append(annotation_row)


[1/784] AL1Y-NB1.annotated.cnr
[25/784] ALA8-NB1.annotated.cnr
[50/784] AM2J-NB1.annotated.cnr
[75/784] AMHD-NB2.annotated.cnr
[100/784] AMT0-NB2.annotated.cnr
[125/784] AMVG-NB2.annotated.cnr
[150/784] AN34-NB2.annotated.cnr
[175/784] ANFU-NB6.annotated.cnr
[200/784] ANOR-NB1.annotated.cnr
[225/784] ANST-NB6.annotated.cnr
[250/784] ANU1-NB8.annotated.cnr
[275/784] AO01-NB5.annotated.cnr
[300/784] AO1Q-NB1.annotated.cnr
[325/784] AO2R-NB8.annotated.cnr
[350/784] AOFA-NB1.annotated.cnr
[375/784] AOHF-NB4.annotated.cnr
[400/784] AOIK-NB2.annotated.cnr
[425/784] AOLF-NB1.annotated.cnr
[450/784] AON9-NB7.annotated.cnr
[475/784] AOXB-NB1.annotated.cnr
[500/784] AOYK-NB2.annotated.cnr
[525/784] APDH-NB1.annotated.cnr
[550/784] APJC-NB1.annotated.cnr
[575/784] APKB-NB2.annotated.cnr
[600/784] APOM-NB1.annotated.cnr
[625/784] APZ6-NB2.annotated.cnr
[650/784] AQ1D-NB5.annotated.cnr
[675/784] AR63-NB3.annotated.cnr
[700/784] ARHA-NB5.annotated.cnr
[725/784] ARMP-NB1.annotated.cnr
[750/784] ARQJ-

In [22]:
# 6. Build gene x sample matrix

# ============================================================

cnv_matrix = pd.DataFrame(gene_state_by_sample)

# Missing gene/sample combinations are assumed neutral because each sample is
# expected to be genome-wide and the all-sample gene universe is the union.
cnv_matrix = cnv_matrix.fillna("NEUT")

cnv_matrix.insert(0, "gene", cnv_matrix.index)
cnv_matrix = cnv_matrix.reset_index(drop=True)

manifest = pd.DataFrame(manifest_rows).sort_values(
    ["Patient_Identifier", "draw_num", "sample_id"],
    na_position="last"
)

sample_order = manifest["sample_id"].tolist()
cnv_matrix = cnv_matrix[["gene"] + sample_order]


In [23]:
# 7. Build manifest and annotation with TFx metadata
# ============================================================

annotation = pd.DataFrame(annotation_rows).sort_values(
    ["Patient_Identifier", "draw_num", "sample_id"],
    na_position="last"
)

annotation = annotation.merge(
    metasheet_annot,
    on="sample_id",
    how="left"
)

# Prefer sample-derived Patient_Identifier, but keep metasheet version for QC.
annotation["Patient_Identifier_metasheet_match"] = (
    annotation["Patient_Identifier"].astype(str)
    == annotation["Patient_Identifier_from_metasheet"].astype(str)
)

# Create columns expected by the downstream clinical/CNV extraction script.
if "Histological_subtype" in annotation.columns:
    annotation["Histology_group"] = annotation["Histological_subtype"]
else:
    annotation["Histology_group"] = pd.NA

if "Molecular_subtype" in annotation.columns:
    annotation["Molecular_subtype_clean"] = annotation["Molecular_subtype"]
else:
    annotation["Molecular_subtype_clean"] = pd.NA

# Rename patient_id column from metasheet if present.
if patient_id_col is not None and patient_id_col in annotation.columns:
    annotation = annotation.rename(columns={patient_id_col: "patient_id_from_metasheet"})

if broad_col is not None and broad_col in annotation.columns:
    annotation = annotation.rename(columns={broad_col: "Broad_entity_sample_id"})

# Reorder annotation columns.
front_cols = [
    "sample_id",
    "Broad_entity_sample_id",
    "patient_id_from_metasheet",
    "Patient_Identifier",
    "Patient_Identifier_from_metasheet",
    "Patient_Identifier_metasheet_match",
    "draw_num",
    "Tumor_Fraction",
    "Tumor_Fraction_percent",
    "Ploidy",
    "Mean_coverage",
    "Histological_subtype",
    "Histology_group",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Molecular_subtype",
    "Molecular_subtype_clean",
    "Subclone Fraction",
    "Frac. Genome Subclonal",
    "Frac. CNA Subclonal",
    "Frac_Subnucleosomal_90_150",
    "Frac_Mononucleosomal_151_250",
    "Frac_Dinucleosomal_251_450",
    "Frac_Trinucleosomal_451_650",
    "CNV_burden_non_NEUT_genes",
    "n_amp_or_hlamp_genes",
    "n_hlamp_genes",
    "CNR_file",
]

front_cols = [c for c in front_cols if c in annotation.columns]
annotation = annotation[front_cols + [c for c in annotation.columns if c not in front_cols]]


In [24]:
# 8. Save outputs
# ============================================================

cnv_matrix.to_csv(out_matrix, sep="\t", index=False)
annotation.to_csv(out_annotation, sep="\t", index=False)
manifest.to_csv(out_manifest, sep="\t", index=False)

print("\n========== Finished ==========")
print("Output directory:")
print(out_dir)

print("\nSaved matrix:")
print(out_matrix)
print("Matrix dimensions:", cnv_matrix.shape)
print("Unique genes:", cnv_matrix.shape[0])
print("Samples:", cnv_matrix.shape[1] - 1)

print("\nSaved sample annotation:")
print(out_annotation)
print("Annotation rows:", annotation.shape[0])
print("Samples with Tumor_Fraction_percent:", annotation["Tumor_Fraction_percent"].notna().sum())

print("\nSaved manifest:")
print(out_manifest)
print("Manifest rows:", manifest.shape[0])

print("\nUnique cases:", manifest["Patient_Identifier"].nunique(dropna=True))

print("\nTop 10 cases by number of samples:")
print(
    manifest
    .groupby("Patient_Identifier", dropna=False)["sample_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print("\nMetasheet matching summary:")
print(annotation["Patient_Identifier_metasheet_match"].value_counts(dropna=False))

missing_tfx = annotation[annotation["Tumor_Fraction_percent"].isna()]
print("\nSamples missing Tumor_Fraction_percent:", missing_tfx.shape[0])
if missing_tfx.shape[0] > 0:
    print(missing_tfx[["sample_id", "Patient_Identifier", "CNR_file"]].head(20))



========== Finished ==========
Output directory:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction

Saved matrix:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_gene_CNV_state_matrix.tsv
Matrix dimensions: (24881, 785)
Unique genes: 24881
Samples: 784

Saved sample annotation:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_CNV_sample_annotation.tsv
Annotation rows: 784
Samples with Tumor_Fraction_percent: 784

Saved manifest:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_CNR_manifest.tsv
Manifest rows: 784

Unique cases: 270

Top 10 cases by number of samples:
Patient_Identifier
ANMI    9
AOFM    9
AL9C    8
ANTU    8
ALA9    8
ANYY    8
ANU1    8
APZS    8
AOGX    8
AON9    8
Name: sample_id, dtype: int64

Metasheet mat